# Full-run test for TVAE

This notebook provides runnable cells to perform a full training run for `TVAE` using the project `TrainTestSplitPipeline`.

Notes:
- The notebook defaults to skipping the pipeline TSTR evaluations (these import `xgboost`) to avoid extra dependency installs. Set `SKIP_EVALUATIONS = False` if you have `xgboost` installed and want full evaluation.
- Adjust the model config dictionaries below to control epochs / steps / batch sizes for real full runs.
- Each cell is annotated so you can run the cells interactively per dataset/model.

In [7]:
pip install ctgan

In [8]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

# Convenience wrapper to create a pipeline that optionally disables evaluations
def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [9]:
pip install xgboost

In [10]:
# User configuration: choose dataset(s) and run options
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']  # change or add: e.g., ['adult','car']
# If True, the pipeline will NOT run TSTR evaluations (avoids needing xgboost)
SKIP_EVALUATIONS = False

# Top-level dirs
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

# Model mapping: name -> (module_path, class_name)
# Model mapping
MODEL_MAP = {
    'ctgan': ('katabatic.models.ctgan.models', 'CTGANModel'),
    'ctabgan': ('katabatic.models.ctabgan.models', 'CTABGANModel'),
    'tvae': ('katabatic.models.tvae.models', 'TVAEModel'),  # NEW
    'tabddpm': ('katabatic.models.tabddpm.models', 'Tabddpm'),
    'tabsyn': ('katabatic.models.tabsyn.models', 'TabSyn'),
}

# TVAE Configuration (from CTGAN library defaults)
TVAE_CONFIG = {
    'epochs': 300,
    'batch_size': 500,
    'compress_dims': (128, 128),
    'decompress_dims': (128, 128),
    'embedding_dim': 128,
    'l2scale': 1e-5,
    'loss_factor': 2,
    'cuda': True,
}

# Default full-run configs for each model (tweak as needed)
CTGAN_CONFIG = {
    'epochs': 200,
    'batch_size': 512,
    'noise_dim': 128,
    'backend': 'torch',
}

TABDDPM_CONFIG = {
    # TabDDPM expects a 'config' passed into train via pipeline; pipeline.run will pass through kwargs to model.train
    'config': {
        'steps': 1000,
        'num_timesteps': 500,
        'batch_size': 64,
        'use_ema': True,
        'd_layers': (128,128),
    }
}

TABSYN_CONFIG = {
    'decoder_epochs': 50,
    'decoder_batch_size': 1024,
    'diffusion_epochs': 200,
    'diffusion_batch_size': 256,
}

## Preprocess datasets (run once)
Run this cell to discretize the raw CSVs into `discretized_data/{dataset}.csv`. 

In [11]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(file_path=f'raw_data/{dataset}.csv', output_path=f'discretized_data/{dataset}.csv', bins=10, strategy='uniform')
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        import traceback; traceback.print_exc()


Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
Discretized -> discretized_data/shuttle.csv


## Run full TVAE
This cell runs CTGAN for each selected dataset using `CTGAN_CONFIG` above. Be patient — full training can take time depending on `epochs` and dataset size.

In [12]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'TVAE -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'tvae')
    ensure(synth_dir)
    try:
        mod_path, cls_name = MODEL_MAP['tvae']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)
        model_factory = lambda: ModelClass(**TVAE_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        print('TVAE finished for', dataset, '-> result:', result)
    except Exception as e:
        print('TVAE failed for', dataset, e)
        import traceback; traceback.print_exc()


TVAE -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[TVAE] Detected discrete columns: ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'class']
[TVAE] Initializing TVAE with 300 epochs...
[TVAE] Training on 26048 samples...
[TVAE] Finished training in 453.53 seconds.
[TVAE] Generating 26048 synthetic samples...
[TVAE] Synthetic data saved:
  X -> synthetic\adult\tvae\x_synth.csv
  y -> synthetic\adult\tvae\y_synth.csv

Results saved to: Results\adult\tvae_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.8139
F1 Score: 0.79

Traceback (most recent call last):
  File "C:\Users\lbrum\AppData\Local\Temp\ipykernel_27468\3748459007.py", line 12, in <module>
    result = pipeline.run(
             ^^^^^^^^^^^^^
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\pipeline\train_test_split\pipeline.py", line 46, in run
    eval_instance.evaluate()
  File "C:\Users\lbrum\OneDrive\Documents\Uni\2025\T3\SIT374\Katabatic\katabatic\evaluate\tstr\evaluation.py", line 63, in evaluate
    model.fit(self.x_train, self.y_train)
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\xgboost\core.py", line 729, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 1640, in fit
    raise ValueError(
ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3], got [0 1 3 4]



TVAE -> shuttle
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
0    0.785970
3    0.153491
4    0.056336
2    0.002953
1    0.000862
6    0.000216
5    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.785948
3    0.153534
4    0.056293
2    0.002931
1    0.000862
6    0.000259
5    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
[TVAE] Detected discrete columns: ['time', 'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'class']
[TVAE] Initializing TVAE with 300 epochs...
[TVAE] Training on 46400 samples...
[TVAE] Finished training in 732.52 seconds.
[TVAE] Generating 46400 synthetic samples...
[TVAE] Synthetic data saved:
  X -> synthetic\shuttle\tvae\x_synth.csv
  y -> synthetic\shuttle\tvae\y_synth.csv

Results saved to: Results\shuttle\tvae_tstr.csv

TSTR Evaluation Results:

LR: